# Day 2, hands-on 2: the truncated feed, worked

A vendor JSON file that stops mid-record, read at the line the parser names.

Every placeholder is filled with the option the answer key records, and the notebook is executed
from a clean kernel so every output and every check is visible on the page. The line under each
step says why the other three letters fail.

Where this sits in the day, and the steps this notebook walks.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["functions and errors", "files and formats", "hands-on: trace the calls", "hands-on: the truncated feed"], lit=3, title="the day's notebooks", show=False),
    kit.flow(["ask JSON to load it", "read the message", "open the named line", "say what is missing"], title="this notebook's steps", show=False),
)

## Setup

The truncated vendor feed in `../data/`, exactly as it arrived. Nothing here is repaired for you.

In [2]:
import json

raw = kit.read_text("C2_W01_D02_vendor_truncated_STUDENT.json")
lines = raw.splitlines()

print(len(lines), "lines")
print(lines[0])
print(lines[-1])

47 lines
[
      "signup_date": "2026-08-10"


## Step 1. Ask JSON to load it

A JSON file is either wholly valid or wholly unreadable. Catch the failure so the notebook keeps running, and read what the parser says.

In [3]:
kit.flow(["ask JSON to load it", "read the message", "open the named line", "say what is missing"], lit=0)

In [4]:
# TODO 1. How many records can you use from a file that failed to parse?
#   a) all of them, minus the last
#   b) none at all
#   c) the 46 complete ones
#   d) it depends on the parser
try:
    data = json.loads(raw)
    message = "it parsed"
except json.JSONDecodeError as e:
    message = str(e)
    data = None

print(message)
half_parsed = 0
print("half parsed records available:", half_parsed)

Expecting ',' delimiter: line 48 column 1 (char 1027)
half parsed records available: 0


In [5]:
kit.check("the file did not parse", data is None)
kit.check("the message names a line and a column", "line" in message and "column" in message,
          message)
kit.check("nothing is usable from a file that failed", half_parsed == 0)

Options a and c describe how a CSV behaves, where each row stands alone. JSON parses the whole document or none of it, so there is no partial result to salvage. Option d suggests the behaviour varies; it does not, because the format itself has no record boundary the parser can stop at safely.

## Step 2. Read the message

The message has three parts and each one is a separate instruction. Take them in order.

In [6]:
kit.flow(["ask JSON to load it", "read the message", "open the named line", "say what is missing"], lit=1)

In [7]:
# TODO 2. What is your first move?
#   a) rewrite the file by hand
#   b) ask the vendor to resend
#   c) open the file at the named line
#   d) switch the parser
import re

named_line = int(re.search(r"line (\d+)", message).group(1))
named_column = int(re.search(r"column (\d+)", message).group(1))
first_move = "open the file at the named line"

print("line", named_line, "column", named_column)
print("first move:", first_move)

line 48 column 1
first move: open the file at the named line


In [8]:
kit.check("the parser named line 48", named_line == 48, f"line {named_line}")
kit.check("the file itself is 47 lines, so the named line is past the end", named_line > len(lines))
kit.check("the first move is to look", "open" in first_move)

Options a and b are what to do after you know what is wrong, and doing either first means you never learn what the vendor got wrong. Option d treats a truthful error as a tool problem, which is the most expensive habit on this list.

## Step 3. Open the named line

The parser named a line past the end of the file. That is not a bug in the message; it is the message telling you the file stopped before it should have.

In [9]:
kit.flow(["ask JSON to load it", "read the message", "open the named line", "say what is missing"], lit=2)

In [10]:
# TODO 3. What does the last line tell you?
#   a) the file ends inside a record
#   b) the file ends with a closing bracket
#   c) the file is empty
#   d) the file ends with a comment
tail = lines[-3:]
for n, line in enumerate(tail, start=len(lines) - 2):
    print(n, repr(line))

ends_cleanly = False
print("ends cleanly:", ends_cleanly)

45 '      "customer_id": "C1184",'
46 '      "city": "Singapore",'
47 '      "signup_date": "2026-08-10"'
ends cleanly: False


In [11]:
kit.check("the file does not end with a closing bracket", not lines[-1].strip().endswith("]"))
kit.check("so it stops inside a record", ends_cleanly is False)

Every option renders the same value, so the letter records what you read off the screen. Option b is what a complete file looks like. Option c is contradicted by 47 lines of content. Option d imports a habit from other formats; JSON has no comments at all.

## Step 4. Say what is missing

Write the sentence you would send back to the vendor. It names the file, the line, and what a complete file would have had.

In [12]:
kit.flow(["ask JSON to load it", "read the message", "open the named line", "say what is missing"], lit=3)

In [13]:
# TODO 4. What ends that sentence?
#   a) "Please resend."
#   b) "The parser is broken."
#   c) "We fixed it on our side."
#   d) the parser message and what it costs
complete_records = raw.count('"order_id"')
report = (
    "The feed has " + str(len(lines)) + " lines and stops inside record "
    + str(complete_records) + ". "
    + "json.load reports " + message + ", so no record in it can be used."
)
print(report)

The feed has 47 lines and stops inside record 3. json.load reports Expecting ',' delimiter: line 48 column 1 (char 1027), so no record in it can be used.


In [14]:
kit.check("the report names the parser's own wording", "line 48" in report)
kit.check("and says what it costs", "no record" in report)
kit.check("the feed opened three records before it stopped", complete_records == 3,
          f"{complete_records} order_id keys in 47 lines")

## What to post

Post one line with the four letters, then the two numbers the parser gave you:

```
1b 2c 3a 4d
line 48, column 1
```

Then the one sentence you would send the vendor.

In [15]:
kit.flow(["ask JSON to load it", "read the message", "open the named line", "say what is missing"], lit=3, title="the notebook, end to end")
kit.check_summary()